In [4]:
import geopandas as gpd
import rasterio
import osmnx as ox
import numpy as np
import pandas as pd
import folium
import requests
import yaml
import ee
from groq import Groq
from dotenv import load_dotenv
import cdsapi
import os

print("All imports successful")
print("Python version:", __import__('sys').version)

All imports successful
Python version: 3.11.7 (main, Dec 15 2023, 18:12:31) [GCC 11.2.0]


In [5]:
import yaml
with open("/srv/THESIS/energy_profiling_thesis/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)
print("Project:", config["project"])
print("Total coordinates:", config["sampling"]["total_coordinates"])
print("Strata:", config["sampling"]["strata"])
print("Groq model:", config["api"]["groq_model"])
print("GEE project:", config["api"]["gee_project"])

Project: {'name': 'energy_profiling_thesis', 'version': '1.0', 'python_version': '3.11'}
Total coordinates: 5250
Strata: {'dense_urban': 750, 'industrial': 750, 'suburban': 750, 'agricultural': 500, 'forest': 500, 'coastal': 500, 'arid': 500, 'alpine': 375, 'informal_settlements': 375, 'water_wetland': 250}
Groq model: llama-3.3-70b-versatile
GEE project: energy-thesis


In [6]:
import os
import rasterio

rasters = {
    "GHSL Built Surface": "/srv/THESIS/energy_profiling_thesis/rasters/ghsl/built_surface/GHS_BUILT_S_E2020_GLOBE_R2023A_54009_100_V1_0.tif",
    "GHSL Building Height": "/srv/THESIS/energy_profiling_thesis/rasters/ghsl/building_height/GHS_BUILT_H_AGBH_E2018_GLOBE_R2023A_54009_100_V1_0.tif",
    "GHSL Population": "/srv/THESIS/energy_profiling_thesis/rasters/ghsl/population/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0.tif",
    "GHSL DEGURBA": "/srv/THESIS/energy_profiling_thesis/rasters/ghsl/degurba/GHS_SMOD_E2020_GLOBE_R2023A_54009_1000_V2_0.tif",
    "Solar Atlas DNI": "/srv/THESIS/energy_profiling_thesis/rasters/global_solar_atlas/World_DNI_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF/DNI.tif",
    "Solar Atlas GHI": "/srv/THESIS/energy_profiling_thesis/rasters/global_solar_atlas/World_GHI_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF/GHI.tif",
    "Solar Atlas PVOUT": "/srv/THESIS/energy_profiling_thesis/rasters/global_solar_atlas/World_PVOUT_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF/PVOUT.tif",
}

for name, path in rasters.items():
    if os.path.exists(path):
        with rasterio.open(path) as src:
            print(f"✅ {name}: CRS={src.crs}, Size={src.width}x{src.height}")
    else:
        print(f"❌ {name}: FILE NOT FOUND")

✅ GHSL Built Surface: CRS=ESRI:54009, Size=360820x180000
✅ GHSL Building Height: CRS=ESRI:54009, Size=360820x180000
✅ GHSL Population: CRS=ESRI:54009, Size=360820x180000
✅ GHSL DEGURBA: CRS=ESRI:54009, Size=36082x18000
✅ Solar Atlas DNI: CRS=EPSG:4326, Size=144000x50000
✅ Solar Atlas GHI: CRS=EPSG:4326, Size=144000x50000
✅ Solar Atlas PVOUT: CRS=EPSG:4326, Size=43200x15000


In [7]:
import requests

query = """
[out:json][timeout:25];
(
  way["power"="line"](51.48,0.0,51.52,0.05);
);
out body;
"""
response = requests.post("https://overpass-api.de/api/interpreter", data={"data": query})
print("Overpass status:", response.status_code)
print("Elements returned:", len(response.json()["elements"]))

Overpass status: 200
Elements returned: 32


In [8]:
import cdsapi
client = cdsapi.Client()
print("CDS client initialized successfully")

CDS client initialized successfully


In [9]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv("/srv/THESIS/energy_profiling_thesis/.env")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say: Groq API working"}],
    max_tokens=20
)
print("Groq:", response.choices[0].message.content)

Groq: Groq API working. Is there anything specific you'd like to know or discuss regarding the Groq


In [10]:
import ee
ee.Initialize(project='energy-thesis')
point = ee.Geometry.Point([13.40, 52.52])
viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')\
    .filterDate('2022-01-01', '2022-12-31')\
    .select('avg_rad')\
    .mean()
value = viirs.sample(point, 500).first().get('avg_rad').getInfo()
print("GEE VIIRS (Berlin):", value)

GEE VIIRS (Berlin): 80.11833190917969


In [11]:
import subprocess
result = subprocess.run(
    ["aws", "s3", "ls", "s3://esa-worldcover/v200/2021/map/", "--no-sign-request"],
    capture_output=True, text=True
)
lines = result.stdout.strip().split('\n')
print(f"ESA WorldCover accessible: {len(lines)} tiles found")
print("Sample tile:", lines[0])

ESA WorldCover accessible: 2651 tiles found
Sample tile: 2022-10-26 14:36:37    2552807 ESA_WorldCover_10m_2021_v200_N00E006_Map.tif


In [12]:
import pandas as pd

df = pd.read_csv("/srv/THESIS/energy_profiling_thesis/data/coordinates/coordinates_5000.csv")
print("Total rows:", len(df))
print("Null values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated(subset=["lat","lon"]).sum())
print("Strata:", df["stratum"].value_counts().to_dict())
print("Lat range:", round(df["lat"].min(),2), "to", round(df["lat"].max(),2))
print("Lon range:", round(df["lon"].min(),2), "to", round(df["lon"].max(),2))
print("Columns:", list(df.columns))

Total rows: 5250
Null values: 0
Duplicates: 0
Strata: {'dense_urban': 750, 'industrial': 750, 'suburban': 750, 'agricultural': 500, 'forest': 500, 'coastal': 500, 'arid': 500, 'alpine': 375, 'informal_settlements': 375, 'water_wetland': 250}
Lat range: -44.97 to 67.91
Lon range: -124.99 to 153.61
Columns: ['id', 'lat', 'lon', 'stratum', 'bbox_min_lat', 'bbox_max_lat', 'bbox_min_lon', 'bbox_max_lon']
